# 07 · Walk-forward por regime de mercado

As etapas 01 a 06 usam um único teste (13/09 a 31/12/2023). Aqui a comparação se repete em 20 meses:
- De maio/2022 a dezembro/2023, em blocos de 2 meses.
- Em cada bloco, a rede é retreinada com todo o passado (os últimos 15% dele param o treino) e prevê o bloco.
- Os dias são agrupados em três regimes: **Queda 2022** (mai–dez/2022), **Lateral 2023** (jan–set/2023) e **Alta fim de 2023** (out–dez/2023).

Mesmas redes, hiperparâmetros e versões da etapa 05 (financeiro, sentimento_ffill e sentimento_btc), com 5 sementes.
São 1.500 treinos (10 vezes a etapa 05). No Kaggle, use **Save Version → Save & Run All** para rodar em segundo plano.

In [ ]:
import os, sys, glob
achados = glob.glob('../src/tcc/config.py') + glob.glob('/kaggle/input/**/src/tcc/config.py', recursive=True)
RAIZ = achados[0].rsplit('/src/', 1)[0]              # pasta do projeto (no PC ou no dataset do Kaggle)
sys.path.insert(0, os.path.join(RAIZ, 'src'))
import warnings; warnings.filterwarnings('ignore')
import json, shutil
import pandas as pd, matplotlib.pyplot as plt
from tcc import config, experimento, sentimento, walkforward

In [ ]:
ALVO   = 'retorno'
INICIO = '2022-01-01'
FIM    = '2023-12-31'
MOEDAS = config.MOEDAS
VARIANTES = {'financeiro': None, 'sentimento_ffill': 'ffill', 'sentimento_btc': 'btc'}

ler_hp = lambda etapa: json.load(open(os.path.join(RAIZ, 'results', f'{etapa}_{ALVO}_{INICIO}_{FIM}', 'melhores_hp.json')))
MODELOS = {'LSTM': ler_hp('02_optuna_lstm'), 'Transformer': ler_hp('03_optuna_transformer')}

SAIDA = config.PASTA_RESULTADOS / f'07_walk_forward_{ALVO}_{INICIO}_{FIM}'
SAIDA.mkdir(parents=True, exist_ok=True)

## Treino e teste

In [ ]:
partes = []
for m in MOEDAS:
    for variante, metodo in VARIANTES.items():
        extras = None if metodo is None else sentimento.carregar(m, metodo, INICIO, FIM)
        for nome, hp in MODELOS.items():
            partes.append(walkforward.rodar(m, ALVO, INICIO, FIM, hp, extras=extras).assign(modelo=nome, variante=variante))
            print(m, variante, nome, 'ok')
    pd.concat(partes, ignore_index=True).to_csv(SAIDA / 'previsoes.csv', index=False)   # guarda a cada moeda
prev = pd.concat(partes, ignore_index=True)

## MAPE ÷ MAPE do naive, por regime

Abaixo de 1, a rede é melhor que o naive.

In [ ]:
ordem = list(config.REGIMES) + ['Todos']
resumo = pd.concat([experimento.resumir(prev, chave=['modelo', 'variante', 'regime']),
                    experimento.resumir(prev.assign(regime='Todos'), chave=['modelo', 'variante', 'regime'])], ignore_index=True)
resumo['razao_naive'] = resumo['MAPE_%'] / resumo['MAPE_naive_%']
resumo.round(4).to_csv(SAIDA / 'resumo.csv', index=False)

tabela = resumo.pivot_table(index=['regime', 'moeda'], columns=['modelo', 'variante'], values='razao_naive')
tabela = tabela.loc[[(r, m) for r in ordem for m in MOEDAS], [(mo, v) for mo in MODELOS for v in VARIANTES]]
tabela.round(3).to_csv(SAIDA / 'tabela_razao_naive.csv')
tabela.groupby(level='regime', sort=False).mean().round(3)

In [ ]:
mape = resumo.pivot_table(index=['regime', 'moeda'], columns=['modelo', 'variante'], values='MAPE_%')
mape = mape.loc[[(r, m) for r in ordem for m in MOEDAS], [(mo, v) for mo in MODELOS for v in VARIANTES]]
mape.insert(0, ('naive', ''), resumo.groupby(['regime', 'moeda'])['MAPE_naive_%'].first())
mape.round(3).to_csv(SAIDA / 'tabela_mape.csv')
mape.round(3)

## Diebold-Mariano: com sentimento x financeiro, por regime

DM < 0: a versão com sentimento erra menos. p < 0,05: diferença significativa.

In [ ]:
dm = pd.concat([experimento.comparar(p, 'financeiro', 'variante', ['sentimento_ffill', 'sentimento_btc'], por=('moeda', 'modelo', 'regime'))
                for p in [prev, prev.assign(regime='Todos')]], ignore_index=True)
dm.round(4).to_csv(SAIDA / 'dm_sentimento_vs_financeiro.csv', index=False)
dm['resultado'] = 'empate'
dm.loc[(dm.p < 0.05) & (dm.DM < 0), 'resultado'] = 'melhor'
dm.loc[(dm.p < 0.05) & (dm.DM > 0), 'resultado'] = 'pior'
pd.crosstab([dm.regime, dm.modelo, dm.variante], dm.resultado).loc[ordem]

## Contra o naive, por regime

Quantas moedas a rede vence ou perde para o naive (p < 0,05).

In [ ]:
resumo['resultado'] = 'empate'
resumo.loc[(resumo.p_vs_naive < 0.05) & (resumo.razao_naive < 1), 'resultado'] = 'vence o naive'
resumo.loc[(resumo.p_vs_naive < 0.05) & (resumo.razao_naive > 1), 'resultado'] = 'perde para o naive'
pd.crosstab([resumo.regime, resumo.modelo, resumo.variante], resumo.resultado).loc[ordem]

In [ ]:
fig, eixos = plt.subplots(1, len(MODELOS), figsize=(12, 4), sharey=True)
media = resumo.groupby(['modelo', 'regime', 'variante'])['razao_naive'].mean()
for ax, nome in zip(eixos, MODELOS):
    for k, v in enumerate(VARIANTES):
        ax.bar([i + (k - 1) * 0.27 for i in range(len(ordem))], [media[(nome, r, v)] for r in ordem], 0.27, label=v)
    ax.axhline(1, color='r', ls='--', lw=1)
    ax.set_xticks(range(len(ordem))); ax.set_xticklabels(ordem, fontsize=8); ax.set_title(nome); ax.grid(alpha=.3, axis='y')
eixos[0].set_ylabel('MAPE ÷ MAPE naive (média das 5 moedas)'); eixos[0].legend(fontsize=8)
lo = media.min(); eixos[0].set_ylim(min(0.95, lo - 0.01), max(media.max() + 0.01, 1.02))
plt.tight_layout(); plt.savefig(SAIDA / 'razao_naive_por_regime.png', dpi=130); plt.show()

In [ ]:
(SAIDA / 'config.json').write_text(json.dumps({'alvo': ALVO, 'inicio': INICIO, 'fim': FIM, 'moedas': MOEDAS,
    'sementes': config.SEMENTES, 'variantes': VARIANTES, 'inicio_teste': config.INICIO_WALK_FORWARD,
    'meses_por_bloco': config.MESES_POR_BLOCO, 'regimes': config.REGIMES, 'hp': MODELOS}, indent=2))
print('Pronto:', shutil.make_archive(str(SAIDA), 'zip', SAIDA))